In [5]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
from pathlib import Path
import platform
import random
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer

from transformers import PreTrainedTokenizerFast


In [6]:
@dataclass(frozen=True)
class WordPiecePaths:
    """Paths used by the QQP WordPiece-tokenizer pipeline."""

    train_csv_path: Path = Path("data/raw/train.csv")
    processed_dir: Path = Path("data/processed")
    tokenizer_dir: Path = Path(
        "artifacts/tokenizers/wordpiece_uncased_30k"
    )

    @property
    def corpus_path(self) -> Path:
        return self.processed_dir / "tokenizer_corpus.txt"

    @property
    def train_split_path(self) -> Path:
        return self.processed_dir / "train_split.csv"

    @property
    def valid_split_path(self) -> Path:
        return self.processed_dir / "valid_split.csv"

    @property
    def tokenizer_json_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer.json"

    @property
    def tokenizer_config_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer_config.json"

    @property
    def special_tokens_map_path(self) -> Path:
        return self.tokenizer_dir / "special_tokens_map.json"

    @property
    def vocab_path(self) -> Path:
        return self.tokenizer_dir / "vocab.txt"

    @property
    def training_metadata_path(self) -> Path:
        return self.tokenizer_dir / "training_metadata.json"

    def make_dirs(self) -> None:
        """Create all output directories."""
        self.processed_dir.mkdir(parents=True, exist_ok=True)
        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)

    def validate_input(self) -> None:
        """Check that the original QQP training CSV exists."""
        if not self.train_csv_path.is_file():
            raise FileNotFoundError(
                f"QQP training CSV was not found: {self.train_csv_path}"
            )


In [7]:
@dataclass
class WordPieceConfig:

    vocab_size: int = 30000
    min_frequency: int = 2
    lowercase: bool = True

    max_sequence_length: int = 64
    max_pair_length: int = 128

    pad_token: str = "[PAD]"
    unk_token: str = "[UNK]"
    cls_token: str = "[CLS]"
    sep_token: str = "[SEP]"
    mask_token: str = "[MASK]"

    valid_size: float = 0.1
    seed: int = 28

    continuing_prefix_subword: str = "##"

    @property
    def special_tokens(self):
        return [
            self.pad_token,
            self.unk_token,
            self.cls_token,
            self.sep_token,
            self.mask_token
        ]

In [12]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

    print(f">>> Seed set to {seed}...")

def save_json(data: Dict[str, Any], path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f">>> JSON saved to {path}...")

def load_json(path: Union[str, Path]) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    print(f"JSON file loaded from {path}...")

In [47]:
def save_json(data: Dict[str, Any], path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f">>> JSON saved to {path}...")

In [48]:
def load_json(path: Union[str, Path]) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        
        return json.load(f)
    print(f"JSON file loaded from {path}...")

In [49]:
def save_tokenizer_config(paths: WordPiecePaths, cfg: WordPieceConfig) -> None:
    config = {
        "paths": {
            "train_csv_path": str(paths.train_csv_path),
            "processed_dir": str(paths.processed_dir),
            "tokenizer_dir": str(paths.tokenizer_dir),
            "corpus_path": str(paths.corpus_path),
            "train_split_path": str(paths.train_split_path),
            "valid_split_path": str(paths.valid_split_path),
            "tokenizer_json_path": str(paths.tokenizer_json_path),
            "artifacts_path": str(paths.artifacts_path)
        },
        "wordpiece": {
            **asdict(cfg),
            "special_tokens": cfg.special_tokens
        },
    }

    save_json(config, paths.configs_path)

In [55]:
def load_qqp_data(csv_path: Union=[str, Path]) -> pd.DataFrame:
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    required_cols = ["question1", "question2", "is_duplicated"]
    missing_cols = [col for col in required_cols if col is not df.columns]
    if missing_cols:
        raise ValueError(f"Missing Required columns: {missing_cols}")

    df = df["required_cols"].copy()

    df["question1"] = df["question1"].fillna("").astype(str)
    df["question2"] = df["question2"].fillna("").astype(str)
    df["is_duplicated"] = df["is_duplicated"].astype(int)

    df = df[
        df["question1"].str.strip() != "" |
        df["question2"].str.strip() != ""
    ].reset_index(drop=True)

    print(
        f">>> QQP data loaded from {path} successfully...\n"
        f">>> DataFrame shape: {df.shape}"
         )
    return df

In [54]:
def create_train_valid_split(
    df: pd.DataFrame,
    cfg: WordPieceConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    train_df, val_df = train_test_split(
        df,
        test_size=cfg.valid_size,
        seed=cfg.seed,
        stratify=df["is_duplicated"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(
        f">>> DataFrame splitted to train and validation...\n"
        f">>> Train Size: {len(train_df)}\n"
        f">>> Validation Size: {len(val_df)}"
    )

    return train_df, val_df

In [56]:
def save_splits(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    paths: WordPiecePaths
) -> None:
    train_df.to_csv(paths.train_split_path, index=False)
    val_df.to_csv(paths.valid_split_path, index=False)

    print(
        f">>> Train DataFrame saved to: {paths.train_split_path}\n"
        f">>> Validation DataFrame saved to: {paths.valid_split_path}"
    )

In [58]:
def build_corpus_file(
    train_df: pd.DataFrame,
    paths: WordPiecePaths
) -> None:
    
    with open(paths.corpus_path, "w", encoding="utf-8") as f:
        for q1, q2 in zip(train_df["question1"], train_df["question2"]):
            q1 = str(q1).strip()
            q2 = str(q2).strip()

            if q1:
                f.write(q1, "\n")
            if q2:
                f.write(q2, "\n")

    print(f">>> Tokenizer corpus saved to: {paths.corpus_path}")

In [8]:
cfg = WordPieceConfig()
paths = WordPiecePaths()

paths.make_dirs()
paths.validate_input()